# Gikomba Market Stall Performance Analysis

**Analyst:** Anne Mugunye · PLP Data Analytics 1 Million Devs Cohort

Wanjiku runs a fresh-produce stall and wants to know how her business is doing, whether it is growing, and what to restock. I use core Python only (no pandas or NumPy).

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Summary statistics
`sum`, `max` and `min` give the totals; `list.index()` finds the best and worst day, and `position // 7 + 1` is the week number.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest, lowest = max(daily_sales), min(daily_sales)
best_pos, worst_pos = daily_sales.index(highest), daily_sales.index(lowest)
best_day, best_week = days[best_pos], best_pos // 7 + 1
worst_day, worst_week = days[worst_pos], worst_pos // 7 + 1
print(f"Total KES {total_sales:,} | Average KES {average_sales:,.2f}")
print(f"Best: {best_day} wk{best_week} KES {highest:,} | Worst: {worst_day} wk{worst_week} KES {lowest:,}")

Total KES 46,450 | Average KES 3,317.86
Best: Sat wk2 KES 5,600 | Worst: Tue wk1 KES 1,950


## Task 2: Weekly totals and growth
`week_totals()` slices the list into week 1 and week 2 and returns both totals as a tuple.

In [3]:
def week_totals(sales):
    """Return (week 1 total, week 2 total) for 14 daily sales."""
    return sum(sales[:7]), sum(sales[7:])


week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100
print(f"Week 1 KES {week1_total:,} | Week 2 KES {week2_total:,} | Growth {growth_pct:.1f}%")

Week 1 KES 22,050 | Week 2 KES 24,400 | Growth 10.7%


## Task 3: Classifying days
`classify_day()` returns Strong (above average × 1.2), Weak (below average × 0.8) or Normal. The dictionary counting pattern totals each label.

In [4]:
def classify_day(amount, average):
    """Return "Strong" (>20% above average), "Weak" (>20% below) or "Normal"."""
    if amount > average * 1.2:
        return "Strong"
    elif amount < average * 0.8:
        return "Weak"
    return "Normal"


day_counts = {}
for amount in daily_sales:
    label = classify_day(amount, average_sales)
    day_counts[label] = day_counts.get(label, 0) + 1
print(day_counts)

{'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekdays versus weekends
I split the sales by day name, average each group, and compare the two averages.

In [5]:
weekday_sales = [s for d, s in zip(days, daily_sales) if d not in ("Sat", "Sun")]
weekend_sales = [s for d, s in zip(days, daily_sales) if d in ("Sat", "Sun")]
weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_lift_pct = (weekend_avg - weekday_avg) / weekday_avg * 100
print(f"Weekday KES {weekday_avg:,.2f} | Weekend KES {weekend_avg:,.2f} | Difference {weekend_lift_pct:.1f}%")

Weekday KES 2,685.00 | Weekend KES 4,900.00 | Difference 82.5%


## Task 5: Reorder list and gross margins
`products_to_reorder()` loops over `stock.items()`; `gross_margin_pct()` applies (price − cost) ÷ price once per product.

In [6]:
def products_to_reorder(items):
    """Return the products whose stock is at or below the reorder level."""
    return [name for name, info in items.items() if info["qty"] <= info["reorder"]]


def gross_margin_pct(info):
    """Return the gross margin percentage: (price - cost) / price * 100."""
    return (info["price"] - info["cost"]) / info["price"] * 100


to_reorder = products_to_reorder(stock)
margins = {name: gross_margin_pct(info) for name, info in stock.items()}
print(to_reorder, {n: round(m, 1) for n, m in margins.items()})

['Tomatoes', 'Sukuma'] {'Tomatoes': 33.3, 'Onions': 30.0, 'Sukuma': 40.0, 'Potatoes': 33.3, 'Avocado': 42.9}


## Task 6: Stall Performance Report
f-strings format the report: `:,` adds thousands separators and `:<10` / `:>5` align the columns.

In [7]:
line = "=" * 46
print(line)
print("      STALL PERFORMANCE REPORT - GIKOMBA")
print(line)
print(f"Total sales:          KES {total_sales:,}")
print(f"Average per day:      KES {average_sales:,.2f}")
print(f"Best day:             {best_day}, week {best_week} (KES {highest:,})")
print(f"Worst day:            {worst_day}, week {worst_week} (KES {lowest:,})")
print(f"Week 1 total:         KES {week1_total:,}")
print(f"Week 2 total:         KES {week2_total:,}")
print(f"Week-on-week growth:  {growth_pct:.1f}%")
print(f"Weekday average:      KES {weekday_avg:,.2f}")
print(f"Weekend average:      KES {weekend_avg:,.2f} (+{weekend_lift_pct:.1f}%)")
print("Days:", ", ".join(f"{k} {day_counts.get(k, 0)}" for k in ("Strong", "Normal", "Weak")))
print(f"{'Product':<10}{'Stock':>7}{'Reorder':>9}{'Margin':>9}  Status")
for name, info in stock.items():
    status = "REORDER" if name in to_reorder else "OK"
    print(f"{name:<10}{info['qty']:>5}kg{info['reorder']:>7}kg{margins[name]:>8.1f}%  {status}")
print(line)

      STALL PERFORMANCE REPORT - GIKOMBA
Total sales:          KES 46,450
Average per day:      KES 3,317.86
Best day:             Sat, week 2 (KES 5,600)
Worst day:            Tue, week 1 (KES 1,950)
Week 1 total:         KES 22,050
Week 2 total:         KES 24,400
Week-on-week growth:  10.7%
Weekday average:      KES 2,685.00
Weekend average:      KES 4,900.00 (+82.5%)
Days: Strong 4, Normal 4, Weak 6
Product     Stock  Reorder   Margin  Status
Tomatoes     18kg     25kg    33.3%  REORDER
Onions       40kg     20kg    30.0%  OK
Sukuma        9kg     15kg    40.0%  REORDER
Potatoes     55kg     30kg    33.3%  OK
Avocado      12kg     10kg    42.9%  OK


## Task 7: Recommendations for Wanjiku
1. **Keep going, the business is growing.** Sales rose from KES 22,050 to KES 24,400 (+10.7%) from week 1 to week 2.
2. **Focus on weekends.** They average KES 4,900 against KES 2,685 on weekdays (+82.5%), and all 4 Strong days were weekends, so stock up before Friday.
3. **Lift the weak weekdays.** 6 of 14 days were Weak (every Monday, Tuesday and Thursday), with a low of KES 1,950, so try a weekday bundle offer.
4. **Restock now.** Tomatoes are at 18 kg (reorder level 25 kg) and Sukuma at 9 kg (level 15 kg).
5. **Push Avocado.** It has the best margin at 42.9% and only 12 kg in stock, while Potatoes (55 kg, 33.3%) are well stocked.